In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os 
df = pd.read_csv("df_metrics.csv")
df_metamodel = pd.read_csv("../meta-model_NEW/meta-model_metrics.csv")

In [2]:
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'

In [10]:
def plot_metrics_subplots(df, offsets_dict, save_path='./figs/'):
    os.makedirs(save_path, exist_ok=True)

    metrics = ['precision', 'recall', 'f1']

    fig, axes = plt.subplots(3, 1, figsize=(6, 9), sharex=True)

    models_order = [
        'Multimodal Transformer',
        'Multimodal BiLSTM',
        'BiLSTM',
        'Random Forest',
        'Transformer',
        'Neural Network',
        'Random Classifier',
        'Meta-model'
    ]

    color_dict = {
        'Random Classifier': plt.cm.tab10(0),
        'Neural Network': plt.cm.tab10(1),
        'Random Forest': plt.cm.tab10(3),
        'BiLSTM': plt.cm.tab10(2),
        'Transformer': plt.cm.tab10(4),
        'Multimodal BiLSTM': plt.cm.tab10(9),
        'Multimodal Transformer': plt.cm.tab10(6),
        'Meta-model': plt.cm.tab10(8),
    }

    k = list(range(1, 11))

    handles = []
    labels = []

    for idx, metric_name in enumerate(metrics):

        ax = axes[idx]
        offsets = offsets_dict[metric_name]
        all_values = []

        for model in models_order:

            if model in df['model'].values:
                model_data = df[df['model'] == model].reset_index(drop=True)
                y = model_data[metric_name].values[:10]
                all_values.extend(y)

                color = color_dict.get(model, 'black')

                line, = ax.plot(k, y, label=model, marker='o', color=color, markersize=3)

                if idx == 0:
                    handles.append(line)
                    labels.append(model.replace("Multimodal ", "M."))

            if model == 'Meta-model':
                meta_value = df_metamodel[metric_name.capitalize()].mean()

                line = ax.axhline(
                    meta_value,
                    color=color_dict.get(model, 'black'),
                    linestyle='--',
                    label='Meta-model'
                )
                if idx == 0:
                    handles.append(line)
                    labels.append('Meta-model')

        # labels de ejes
        ax.set_ylabel(metric_name.capitalize(), fontname='Arial')
        ax.grid(True)

        ymin = min(all_values)
        ymax = max(all_values)

        step = 0.05
        yticks = np.arange(
            np.floor(ymin * 100) / 100,
            np.ceil(ymax * 100) / 100 + step,
            step
        )

        ax.set_yticks(yticks)

        ax.set_xlim(0.9, 10.1)
        ax.set_xticks(range(1, 11))

    # xlabel SOLO en el último
    axes[-1].set_xlabel("Number of predictions", fontname='Arial')

    # leyenda arriba
    fig.legend(
        handles,
        labels,
        loc='upper center',
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 1.05)
    )

    plt.tight_layout(rect=[0, 0.08, 1, 1])

    plt.style.use("default")
    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.color": "#d0d0d0",
        "grid.linewidth": 0.6,
        "grid.alpha": 0.3,
        "font.size": 8,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "lines.linewidth": 1.2
    })
    plt.subplots_adjust(hspace=0.15)
    plt.savefig(f"{save_path}metrics_por_k.pdf", format="pdf", bbox_inches="tight")
    plt.savefig(f"{save_path}metrics_por_k.png", bbox_inches="tight")

    plt.close()

offsetsP = {
        'Multimodal Transformer': 0,
        'Multimodal BiLSTM': 0,
        'BiLSTM': -0.003,
        'Random Forest': 0.003,
        'Transformer': -0.003,
        'Neural Network': 0,
        'Random Classifier': 0,
        'Meta-model': 0
    }

offsetsR = {
        'Multimodal Transformer': -0.005,
        'Multimodal BiLSTM': 0.005,
        'BiLSTM': 0,
        'Random Forest': 0,
        'Transformer': 0,
        'Neural Network': 0,
        'Random Classifier': 0,
        'Meta-model': 0
    }

offsetsF = {
        'Multimodal Transformer': -0.004,
        'Multimodal BiLSTM': 0.004,
        'BiLSTM': 0,
        'Random Forest': 0.0025,
        'Transformer': -0.002,
        'Neural Network': 0,
        'Random Classifier': 0,
        'Meta-model': 0
    }
offsets_dict = {
    'precision': offsetsP,
    'recall': offsetsR,
    'f1': offsetsF
}

plot_metrics_subplots(df, offsets_dict)